# 🏋️ Yoga Pose Classifier — Training Notebook

This notebook trains a dense neural network on MoveNet keypoint embeddings to classify yoga poses.

**Pipeline overview**
1. Load & inspect the CSV dataset
2. Preprocess landmarks → normalized pose embeddings
3. Build and train the model
4. Visualize training history
5. Evaluate on the test set
6. Export to TensorFlow.js

## 1  Imports

In [ ]:
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflowjs as tfjs

from data import BodyPart

print('TensorFlow:', tf.__version__)

ModuleNotFoundError: No module named 'tensorflow'

## 2  Configuration

In [ ]:
TRAIN_CSV      = 'train_data.csv'
TEST_CSV       = 'test_data.csv'
CHECKPOINT_PATH = 'weights.best.hdf5'
TFJS_MODEL_DIR  = 'model'

EPOCHS      = 200
BATCH_SIZE  = 16
VAL_SPLIT   = 0.15
PATIENCE    = 20

## 3  Data Loading & Inspection

In [ ]:
def load_csv(csv_path):
    """Load a pose CSV, return features (X), one-hot labels (y), and class names."""
    df = pd.read_csv(csv_path)
    df.drop(['filename'], axis=1, inplace=True)
    classes = df.pop('class_name').unique()
    y = df.pop('class_no')

    X = df.astype('float64')
    y = keras.utils.to_categorical(y)
    return X, y, classes

In [ ]:
X, y, class_names = load_csv(TRAIN_CSV)

print(f'Training samples : {len(X)}')
print(f'Feature columns  : {X.shape[1]}')
print(f'Classes ({len(class_names)})    : {list(class_names)}')
X.head()

In [ ]:
# Class distribution in training CSV
raw_df = pd.read_csv(TRAIN_CSV)
counts = raw_df['class_name'].value_counts()

fig, ax = plt.subplots(figsize=(10, 4))
counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Training samples per pose class')
ax.set_xlabel('Pose class')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4  Landmark Preprocessing

Raw (x, y, score) keypoints are:
- Centered on the hip midpoint
- Scaled by pose size (max of torso length × 2.5 and max limb reach)
- Flattened to a 34-d embedding (x, y only — scores dropped)

In [ ]:
def get_center_point(landmarks, left_bodypart, right_bodypart):
    """Midpoint between two landmarks."""
    left  = tf.gather(landmarks, left_bodypart.value,  axis=1)
    right = tf.gather(landmarks, right_bodypart.value, axis=1)
    return left * 0.5 + right * 0.5


def get_pose_size(landmarks, torso_size_multiplier=2.5):
    """Normalisation scale: max(torso × multiplier, max limb reach)."""
    hips_center      = get_center_point(landmarks, BodyPart.LEFT_HIP,      BodyPart.RIGHT_HIP)
    shoulders_center = get_center_point(landmarks, BodyPart.LEFT_SHOULDER, BodyPart.RIGHT_SHOULDER)
    torso_size       = tf.linalg.norm(shoulders_center - hips_center)

    pose_center = get_center_point(landmarks, BodyPart.LEFT_HIP, BodyPart.RIGHT_HIP)
    pose_center = tf.expand_dims(pose_center, axis=1)
    pose_center = tf.broadcast_to(pose_center, [tf.size(landmarks) // (17 * 2), 17, 2])

    d        = tf.gather(landmarks - pose_center, 0, axis=0, name='dist_to_pose_center')
    max_dist = tf.reduce_max(tf.linalg.norm(d, axis=0))
    return tf.maximum(torso_size * torso_size_multiplier, max_dist)


def normalize_pose_landmarks(landmarks):
    """Translate pose to origin and scale to unit size."""
    pose_center = get_center_point(landmarks, BodyPart.LEFT_HIP, BodyPart.RIGHT_HIP)
    pose_center = tf.expand_dims(pose_center, axis=1)
    pose_center = tf.broadcast_to(pose_center, [tf.size(landmarks) // (17 * 2), 17, 2])
    landmarks   = landmarks - pose_center
    landmarks  /= get_pose_size(landmarks)
    return landmarks


def landmarks_to_embedding(landmarks_and_scores):
    """Reshape flat 51-d input → normalize → flatten to 34-d embedding."""
    reshaped  = keras.layers.Reshape((17, 3))(landmarks_and_scores)
    landmarks = normalize_pose_landmarks(reshaped[:, :, :2])
    return keras.layers.Flatten()(landmarks)


def preprocess_data(X):
    out = []
    for i in range(X.shape[0]):
        emb = landmarks_to_embedding(
            tf.reshape(tf.convert_to_tensor(X.iloc[i]), (1, 51))
        )
        out.append(tf.reshape(emb, (34,)))
    return tf.convert_to_tensor(out)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=VAL_SPLIT, random_state=42)
X_test,  y_test,  _            = load_csv(TEST_CSV)

print(f'Train: {len(X_train)}  |  Val: {len(X_val)}  |  Test: {len(X_test)}')

In [ ]:
print('Preprocessing training data...')
processed_X_train = preprocess_data(X_train)
print('Preprocessing validation data...')
processed_X_val   = preprocess_data(X_val)
print('Preprocessing test data...')
processed_X_test  = preprocess_data(X_test)
print('Done.')

## 5  Model Definition

In [ ]:
inputs  = tf.keras.Input(shape=(34,))
layer   = keras.layers.Dense(128, activation=tf.nn.relu6)(inputs)
layer   = keras.layers.Dropout(0.5)(layer)
layer   = keras.layers.Dense(64,  activation=tf.nn.relu6)(layer)
layer   = keras.layers.Dropout(0.5)(layer)
outputs = keras.layers.Dense(len(class_names), activation='softmax')(layer)

model = keras.Model(inputs, outputs)
model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## 6  Training

In [ ]:
checkpoint    = keras.callbacks.ModelCheckpoint(
    CHECKPOINT_PATH, monitor='val_accuracy',
    verbose=1, save_best_only=True, mode='max'
)
earlystopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=PATIENCE
)

history = model.fit(
    processed_X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(processed_X_val, y_val),
    callbacks=[checkpoint, earlystopping]
)

## 7  Training History Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.suptitle('Training History', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8  Evaluation on Test Set

In [ ]:
loss, accuracy = model.evaluate(processed_X_test, y_test, verbose=0)
print(f'Test Loss     : {loss:.4f}')
print(f'Test Accuracy : {accuracy:.4f}  ({accuracy*100:.2f}%)')

### Confusion Matrix

In [ ]:
y_pred      = model.predict(processed_X_test)
y_pred_cls  = np.argmax(y_pred,  axis=1)
y_true_cls  = np.argmax(y_test,  axis=1)

cm = confusion_matrix(y_true_cls, y_pred_cls)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=class_names, yticklabels=class_names, ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Test Set')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Per-class Report

In [ ]:
print(classification_report(y_true_cls, y_pred_cls, target_names=class_names))

## 9  Export to TensorFlow.js

In [ ]:
tfjs.converters.save_keras_model(model, TFJS_MODEL_DIR)
print(f'TF.js model saved to: {TFJS_MODEL_DIR}')